# M - Model: Pemodelan Klasifikasi Status Desa IDM 2024
## Kelompok 3 - II4013 Data Analytics

Notebook ini berfokus pada pembangunan model klasifikasi terbimbing (*supervised learning*) untuk memprediksi status perkembangan desa (`STATUS_IDM_2024`) berdasarkan dimensi indeks komposit dan fitur-fitur baru hasil *feature engineering*.

### Alur Pengerjaan:
1. **Load Data**: Membaca dataset dari `data/processed/idm_2024_modeling.csv`.
2. **Data Preparation**: Mengodekan variabel target dan membagi data menjadi set pelatihan dan pengujian (80-20 split).
3. **Decision Tree Classifier**: Membangun model pohon keputusan dasar untuk interpretabilitas.
4. **Random Forest Classifier**: Membangun model Random Forest untuk performa klasifikasi yang lebih kuat.
5. **Evaluasi Model**: Menganalisis metrik performa (Accuracy, Precision, Recall, Confusion Matrix).
6. **Feature Importance (RQ3)**: Menganalisis pilar indeks dan fitur mana yang paling menentukan klasifikasi status desa.
7. **Export Model**: Menyimpan model klasifikasi terbaik ke berkas `.pkl` untuk Streamlit.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Konfigurasi grafik
pd.set_option('display.max_columns', 50)
plt.rcParams.update({
    'figure.dpi': 120,
    'figure.figsize': (10, 5),
    'axes.titlesize': 13,
    'axes.labelsize': 11
})
sns.set_theme(style='whitegrid', palette='muted')

print('[OK] Library siap.')

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())
PROCESSED_DATA_PATH = os.path.join(BASE_DIR, 'data', 'processed', 'idm_2024_modeling.csv')
MODELS_DIR = os.path.join(BASE_DIR, 'models')

df = pd.read_csv(PROCESSED_DATA_PATH, dtype={'KODE_DESA': str})
print(f'Memuat dataset: {df.shape[0]:,} baris x {df.shape[1]} kolom')
df.head(3)

In [ ]:
# 1. Memilih fitur numerik dan fitur kategori ter-encode
fitur_numerik = ['IKS_2024', 'IKE_2024', 'IKL_2024', 'jumlah_rekomendasi', 'total_nilai_rekomendasi', 'gap_iks_ike', 'gap_iks_ikl', 'intensitas_rekomendasi']

# Encode fitur 'dimensi_terendah' (One-Hot Encoding)
df_encoded = pd.get_dummies(df, columns=['dimensi_terendah'], drop_first=False)
fitur_tambahan = [c for c in df_encoded.columns if c.startswith('dimensi_terendah_')]

X = df_encoded[fitur_numerik + fitur_tambahan]

# 2. Mengodekan target STATUS_IDM_2024 secara terurut (Label Encoding)
# Order: Sangat Tertinggal -> Tertinggal -> Berkembang -> Maju -> Mandiri
target_order = ['SANGAT TERTINGGAL', 'TERTINGGAL', 'BERKEMBANG', 'MAJU', 'MANDIRI']
le = LabelEncoder()
le.fit(target_order)
y = le.transform(df['STATUS_IDM_2024'])

# 3. Split data training & testing (80-20) dengan stratifikasi target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)

print('Tipe dan dimensi data training/testing:')
print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')
print(f'Fitur yang digunakan: {X.columns.tolist()}')
print(f'Kelas target: {le.classes_}')

In [ ]:
# Membangun Decision Tree Classifier
print('Melatih model Decision Tree...')
dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
dt_model.fit(X_train, y_train)

# Prediksi & Evaluasi
y_pred_dt = dt_model.predict(X_test)
acc_dt = accuracy_score(y_test, y_pred_dt)
print(f'Accuracy Decision Tree: {acc_dt:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_dt, target_names=le.classes_))

# Visualisasi Pohon Keputusan Sederhana
plt.figure(figsize=(20, 10))
plot_tree(dt_model, feature_names=X.columns.tolist(), class_names=le.classes_.tolist(), filled=True, max_depth=3, fontsize=9)
plt.title('Visualisasi Pohon Keputusan IDM (Maks. Kedalaman = 3)')
plt.tight_layout()
plt.show()

In [ ]:
# Membangun Random Forest Classifier
print('Melatih model Random Forest...')
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train, y_train)

# Prediksi & Evaluasi
y_pred_rf = rf_model.predict(X_test)
acc_rf = accuracy_score(y_test, y_pred_rf)
print(f'Accuracy Random Forest: {acc_rf:.4f}')
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

In [ ]:
# Membuat perbandingan Confusion Matrix
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm_dt = confusion_matrix(y_test, y_pred_dt)
cm_rf = confusion_matrix(y_test, y_pred_rf)

sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[0])
axes[0].set_title('Confusion Matrix: Decision Tree')
axes[0].set_xlabel('Prediksi')
axes[0].set_ylabel('Sebenarnya')

sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[1])
axes[1].set_title('Confusion Matrix: Random Forest')
axes[1].set_xlabel('Prediksi')
axes[1].set_ylabel('Sebenarnya')

plt.tight_layout()
plt.show()

In [ ]:
# Menganalisis Feature Importance dari model Random Forest (untuk menjawab RQ3)
importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]
features = X.columns

df_importance = pd.DataFrame({
    'FITUR': [features[i] for i in indices],
    'IMPORTANCE': [importances[i] for i in indices]
})

print('--- Ranking Kontribusi Fitur Terhadap Status Desa ---')
display(df_importance)

# Plotting Feature Importance
plt.figure(figsize=(10, 6))
sns.barplot(data=df_importance, x='IMPORTANCE', y='FITUR', palette='viridis')
plt.title('Feature Importance dalam Penentuan Status IDM Desa (Random Forest)')
plt.xlabel('Nilai Kontribusi Fitur')
plt.ylabel('Fitur')
plt.tight_layout()
plt.show()

In [ ]:
# Membuat folder models jika belum ada
os.makedirs(MODELS_DIR, exist_ok=True)

# Menyimpan Label Encoder dan Model Random Forest Terbaik
le_pkl_path = os.path.join(MODELS_DIR, 'label_encoder_classification.pkl')
classification_pkl_path = os.path.join(MODELS_DIR, 'classification_model.pkl')

with open(le_pkl_path, 'wb') as f:
    pickle.dump(le, f)

with open(classification_pkl_path, 'wb') as f:
    pickle.dump(rf_model, f)

print(f'[SUCCESS] Label Encoder disimpan ke: {le_pkl_path}')
print(f'[SUCCESS] Model Klasifikasi Random Forest disimpan ke: {classification_pkl_path}')